In [5]:
from pathlib import Path
import mne

EEG_PATH = Path(
    "/Users/chinmay/.Trash/eeg/data/raw/eeg/chb01_01.edf"
)

print("Exists:", EEG_PATH.exists())
print("Size (MB):", EEG_PATH.stat().st_size / (1024 * 1024))

raw = mne.io.read_raw_edf(
    EEG_PATH,
    preload=True,
    verbose=False
)

print("\nEEG LOADED SUCCESSFULLY")
print("Sampling frequency:", raw.info["sfreq"])
print("Number of channels:", len(raw.ch_names))
print("Duration:", raw.times[-1], "seconds")
print("\nChannels:")
print(raw.ch_names)

Exists: True
Size (MB): 1.5308990478515625


/var/folders/tg/4x58h5qj6xb2kptw8fgc0cx40000gn/T/ipykernel_47543/3784749951.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
/var/folders/tg/4x58h5qj6xb2kptw8fgc0cx40000gn/T/ipykernel_47543/3784749951.py:11: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(



EEG LOADED SUCCESSFULLY
Sampling frequency: 256.0
Number of channels: 23
Duration: 134.99609375 seconds

Channels:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'T8-P8-0', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']


In [6]:
from pathlib import Path

# Search the project for the EEG file
project_root = Path("..").resolve()

matches = list(project_root.rglob("chb01_01.edf"))

print("Project root:", project_root)
print("\nMatches found:")

for path in matches:
    print(path)

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning

Matches found:


In [3]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path

EEG_PATH = Path("../data/raw/eeg/chb01_01.edf")

raw = mne.io.read_raw_edf(
    EEG_PATH,
    preload=True,
    verbose=False
)

print("EEG loaded successfully")
print("Sampling frequency:", raw.info["sfreq"])
print("Number of channels:", len(raw.ch_names))
print("Duration:", raw.times[-1], "seconds")

print("\nChannels:")
print(raw.ch_names)

FileNotFoundError: File does not exist: "/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/notebooks/../data/raw/eeg/chb01_01.edf"

In [ ]:
# Inspect first few seconds

data = raw.get_data()

print("Raw EEG shape:", data.shape)
print("Minimum:", data.min())
print("Maximum:", data.max())
print("Mean:", data.mean())
print("Std:", data.std())

In [ ]:
from scipy.signal import welch

# Use first EEG channel
eeg = data[0]
fs_eeg = raw.info["sfreq"]

# First 60 seconds
duration = min(60, len(eeg) / fs_eeg)

eeg_segment = eeg[:int(duration * fs_eeg)]

freqs, psd = welch(
    eeg_segment,
    fs=fs_eeg,
    nperseg=min(1024, len(eeg_segment))
)

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45)
}

features = {}

for band, (low, high) in bands.items():
    mask = (freqs >= low) & (freqs < high)

    power = np.trapezoid(
        psd[mask],
        freqs[mask]
    )

    features[f"{band}_power"] = power

features["eeg_mean"] = np.mean(eeg_segment)
features["eeg_std"] = np.std(eeg_segment)

eeg_features = pd.DataFrame([features])

display(eeg_features)

In [ ]:
output_dir = Path("../data/processed/mimic3")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "eeg_features_demo.csv"

eeg_features.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", eeg_features.shape)